# YbYráSTAC — Pipeline Piloto (MSPA-MA)

Este notebook demonstra o fluxo completo:
1. Instalar YbYráSTAC
2. Descobrir coleções no catálogo
3. Carregar MSPA-Maranhão cloud-native (sem download)
4. Recortar para uma área de interesse
5. Calcular área de Core forest por ano
6. Visualizar série temporal

**Catalog URL:** `https://data.source.coop/celsohlsj/ybyra-br/catalog.json`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/celsohlsj/ybyrastac/blob/main/examples/YbYraSTAC_pipeline_piloto.ipynb)

## 1. Instalação

In [ ]:
# Instala o pacote diretamente do GitHub
!pip install -q 'git+https://github.com/celsohlsj/ybyrastac.git'
!pip install -q geopandas matplotlib contextily

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from ybyrastac import DiscoveryProvider, COGProvider, subset
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd

CATALOG = 'https://data.source.coop/celsohlsj/ybyra-br/catalog.json'
print('✓ Importações OK')

## 2. Descoberta do catálogo

In [ ]:
disc = DiscoveryProvider(CATALOG)
print('Temas disponíveis:', disc.list_themes())
print()
print('Coleções de fragmentação:')
for c in disc.list_collections(theme='fragmentation'):
    print(f"  {c['id']:40s} {c['title']}")

In [ ]:
# Detalha a coleção MSPA-MA
import json
meta = disc.describe('ybyra-mspa-ma')
print(json.dumps({
    'id': meta['id'],
    'extent': meta['extent'],
    'summaries': meta.get('summaries', {})
}, indent=2))

## 3. Carregamento cloud-native (sem download)

In [ ]:
cog = COGProvider(CATALOG)

# Carrega série completa 1985-2023 lazy (via Dask + rioxarray)
ds = cog.open_dataset(
    'ybyra-mspa-ma',
    version='1.0',
    years=list(range(1985, 2024)),
    chunks={'x': 2048, 'y': 2048}
)
print(ds)
print(f'\nResolução: {ds.rio.resolution()} m')
print(f'CRS: {ds.rio.crs}')

## 4. Recorte espacial — Baixada Maranhense

In [ ]:
# Polígono da Baixada Maranhense (municípios selecionados)
# Substitua por qualquer shapefile/geojson de interesse
from shapely.geometry import box

# Bbox aproximado da Baixada Maranhense
aoi = box(-45.8, -3.8, -44.5, -2.7)

ds_aoi = subset(ds, geometry=aoi, crs='EPSG:4674')
print(f'Dimensões após recorte: {ds_aoi.dims}')

## 5. Série temporal de área Core (classe 1)

In [ ]:
# Classe 1 = Core forest (MSPA)
PIXEL_AREA_HA = (30 * 30) / 10_000  # 0.09 ha por pixel a 30m

var_name = [v for v in ds_aoi.data_vars][0]

records = []
for t in ds_aoi.time:
    arr = ds_aoi[var_name].sel(time=t).compute()
    year = int(str(t.values)[:4])
    records.append({
        'year': year,
        'core_ha':        float((arr == 1).sum())  * PIXEL_AREA_HA,
        'edge_ha':        float((arr == 9).sum())  * PIXEL_AREA_HA,
        'islet_ha':       float((arr == 3).sum())  * PIXEL_AREA_HA,
        'background_ha':  float((arr == 0).sum())  * PIXEL_AREA_HA,
    })

df = pd.DataFrame(records).set_index('year')
df['total_forest_ha'] = df['core_ha'] + df['edge_ha'] + df['islet_ha']
df['core_pct'] = df['core_ha'] / df['total_forest_ha'] * 100
print(df.round(0).to_string())

## 6. Visualização

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Painel 1: série temporal de área
ax = axes[0]
df[['core_ha', 'edge_ha', 'islet_ha']].div(1000).plot(
    ax=ax, kind='area', stacked=True,
    color=['#1a7a4a', '#f5a623', '#d62728'],
    alpha=0.85
)
ax.set_title('Área florestal por classe MSPA\nBaixada Maranhense (1985–2023)', fontsize=11)
ax.set_xlabel('Ano')
ax.set_ylabel('Área (× 10³ ha)')
ax.legend(['Core', 'Edge', 'Islet'])

# Painel 2: % core ao longo do tempo
ax2 = axes[1]
ax2.plot(df.index, df['core_pct'], color='#1a7a4a', lw=2, marker='o', ms=4)
ax2.axhline(df['core_pct'].mean(), ls='--', color='gray', lw=1,
            label=f"Média = {df['core_pct'].mean():.1f}%")
ax2.set_title('Proporção de Core forest (%)\nBaixada Maranhense', fontsize=11)
ax2.set_xlabel('Ano')
ax2.set_ylabel('Core (% da área florestal)')
ax2.legend()

plt.tight_layout()
plt.savefig('mspa_baixada_maranhense.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ figura salva')

## 7. Mapa MSPA — ano selecionado

In [ ]:
# Colormap MSPA padrão GuidosToolbox
MSPA_CMAP = mcolors.ListedColormap([
    '#000000',  # 0  background
    '#006400',  # 1  core
    '#ffffff',  # 2  (unused)
    '#ffff00',  # 3  islet
    '#ffffff',  # 4  (unused)
    '#ff69b4',  # 5  perforation
    '#ffffff',  # 6  (unused)
    '#ffffff',  # 7  (unused)
    '#ffffff',  # 8  (unused)
    '#ff8c00',  # 9  edge
])

YEAR = 2023
arr2023 = ds_aoi[var_name].sel(time=str(YEAR), method='nearest').compute()

fig, ax = plt.subplots(figsize=(8, 7))
arr2023.plot(
    ax=ax,
    cmap=MSPA_CMAP,
    vmin=0, vmax=9,
    add_colorbar=False
)
# Legenda manual
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#006400', label='Core'),
    Patch(facecolor='#ff8c00', label='Edge'),
    Patch(facecolor='#ffff00', label='Islet'),
    Patch(facecolor='#ff69b4', label='Perforation'),
    Patch(facecolor='#000000', label='Background'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)
ax.set_title(f'MSPA — Baixada Maranhense {YEAR}', fontsize=12)
plt.tight_layout()
plt.savefig(f'mspa_map_{YEAR}.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Exportar CSV de métricas

Útil para integrar com os dados de métricas de paisagem (`landscapemetrics` no R).

In [ ]:
df.to_csv('mspa_baixada_maranhense_metrics.csv')
print('✓ CSV exportado')
df.round(1)

---
**YbYráSTAC** — IPAM / UFMA / GCBC  
Celso H. L. Silva Junior • https://github.com/celsohlsj/ybyrastac  
Dados: CC-BY-4.0 • Código: EUPL-1.2